# 01_momentum

End-to-end V0 momentum experiment with VNQuant modules.

In [ ]:
import pandas as pd
from config import START_DATE, END_DATE, MOMENTUM_WINDOW, TOP_QUANTILE, FEES, SLIPPAGE, INITIAL_CASH, REBALANCE_FREQUENCY
from src.features.momentum import momentum
from src.signals.ranking import cross_sectional_rank
from src.strategies.long_only import top_quantile
from src.portfolio.weighting import equal_weight, rebalance_weights
from src.backtest.engine import run_backtest

# Replace this block with VNDataProvider(fetcher=...)
# once vnstock is available in your environment.
close_csv = pd.read_csv('src/data/stock_data.csv', parse_dates=True, index_col=0)
close = close_csv.dropna()

signal = cross_sectional_rank(momentum(close, window=MOMENTUM_WINDOW))
selection = top_quantile(signal, q=TOP_QUANTILE)
weights = equal_weight(selection)
weights = rebalance_weights(weights, REBALANCE_FREQUENCY)

print('Selection shape:', selection.shape)
print('Non-zero weight columns:', (weights.sum(axis=0) > 0).sum())


In [ ]:
portfolio, report = run_backtest(
    close=close,
    weights=weights,
    fees=FEES,
    slippage=SLIPPAGE,
    initial_cash=INITIAL_CASH,
    frequency=REBALANCE_FREQUENCY,
)

print(report)
